#Ingestion de archivo "Movie.csv"



###Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark

In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

### En el siguiente caso, definimos el tipo de dato con StructField

In [0]:
# StructType, definimos el esquema para cada tipo de dato
movie_schema = StructType ( fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
] )

movie_df = spark.read \
    .option("header", True) \
    .schema(movie_schema) \
    .csv(f"{bronze_folder_path}/movie.csv")

##Paso 2 - Seleccionar las columnas que se requieren

In [0]:
movies_selected_df = movie_df.select(col("movieId"), col("title"), col("budget"), col("popularity"), col("yearReleaseDate"), col("releaseDate"), col("revenue"), col("durationTime"), col("voteAverage"), col("voteCount").alias("vote_count"))

##Paso 3 -Renombrar columnas

In [0]:
movies_renamed_df = movies_selected_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("yearReleaseDate", "year_release_date")\
    .withColumnRenamed("releaseDate", "release_date")\
    .withColumnRenamed("durationTime", "duration_time")\
    .withColumnRenamed("voteAverage", "vote_average")

##Paso 4 - Añadir columnas a una tabla

In [0]:
movie_final_df = add_ingestion_date(movies_renamed_df)
movie_final_df = add_env(movie_final_df)


In [0]:
display(movie_final_df)

## Paso 5.1 - Guardar en formato parquet pero particionando por el campo year_release_date

In [0]:
#movie_final_df.write.mode("overwrite").format("parquet").save(f"{silver_folder_path}/movies")

#Actualizacion para guardar en tabla
movie_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies")

#df = spark.read.parquet(f"{silver_folder_path}/movies")
#display(df)

In [0]:
dbutils.notebook.exit("El notebook 01.Ingestion File Movie, termino correctamente")